In [1]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

# Define Population Totals
# These totals are fixed for all tests, representing the entire dataset.
TOTAL_ACTIVE = 61471
TOTAL_INACTIVE = 40404
GRAND_TOTAL = TOTAL_ACTIVE + TOTAL_INACTIVE

print(f"--- Fisher's Exact Test Analysis ---")
print(f"Population Baseline: Active={TOTAL_ACTIVE}, Inactive={TOTAL_INACTIVE}\n")

# Load Functional Group Data
# Data extracted from the table: [Functional Group, Active Cases (a), Inactive Cases (b)]
data = [
    ['imidazo', 1050, 126],
    ['ethenyl', 591, 85],
    ['trimethoxyphenyl', 731, 119],
    ['quinolin', 1785, 299],
    ['piperazin(e)*', 4211, 870],
    ['tetrahydro', 1259, 302],
    ['benzothiazol', 1375, 368],
    ['sulfonyl', 1512, 427],
    ['piperidine', 1839, 547],
    ['dimethoxyphenyl', 2647, 806],
    ['methanone', 1831, 596],
    ['phenyl', 8524, 2941],
    ['piperidin', 2457, 851],
    ['trifluoromethyl', 2406, 865],
    ['benzyl', 2458, 892],
    ['methoxyphenyl', 6966, 2559],
    ['methylphenyl', 5961, 2383],
    ['chlorophenyl', 4749, 2004],
    ['ethyl', 8252, 3752],
    ['methyl', 20205, 9283],
    ['carboxamide', 9495, 4910],
    ['oxoethyl', 2420, 3059],
    ['acetate', 655, 982],
    ['carbohydrazide', 126, 460]
]

results = []

# Loop and Perform Fisher's Exact Test 
for group, a, b in data:
    # a: Active cases WITH the group
    # b: Inactive cases WITH the group

    # c: Active cases WITHOUT the group (Total Active - Active with group)
    c = TOTAL_ACTIVE - a

    # d: Inactive cases WITHOUT the group (Total Inactive - Inactive with group)
    d = TOTAL_INACTIVE - b

    # Check for negative values (this should not happen with the provided data,
    # but is a good practice if your 'a' or 'b' exceed the totals)
    if c < 0 or d < 0:
        print(f"Warning: Data error for {group}. Skipping.")
        continue

    # Construct the 2x2 contingency table:
    # Table structure: [[Active_With, Inactive_With], [Active_Without, Inactive_Without]]
    table = np.array([[a, b], [c, d]])

    # Perform Fisher's Exact Test (two-sided alternative is standard)
    # Note: Fisher's Exact Test is preferred for small sample sizes, but it
    # provides an exact p-value even for large counts like these, where a
    # Chi-Squared test would also be acceptable.
    odds_ratio, p_value = fisher_exact(table, alternative='two-sided')

    # Calculate the Ratio (Active/Inactive) for the group for comparison
    ratio_group = a / b if b != 0 else np.inf

    # Store results
    results.append({
        'Functional Group': group,
        'Active (a)': a,
        'Inactive (b)': b,
        'Odds Ratio': odds_ratio,
        'P-value': p_value
    })

# Display Results
df = pd.DataFrame(results)

# Clean up P-value display for very small numbers
df['P-value'] = df['P-value'].apply(lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")

# Sort by Odds Ratio to see the strongest associations first
df_sorted = df.sort_values(by='Odds Ratio', ascending=False).reset_index(drop=True)

# Add a significance column (using a strict Bonferroni-corrected alpha for 24 tests)
# Bonferroni alpha = 0.05 / 24 ≈ 0.00208
SIGNIFICANCE_THRESHOLD = 0.0021

def determine_significance(p_val_str):
    try:
        p_val = float(p_val_str)
        if p_val < SIGNIFICANCE_THRESHOLD:
            return '*** Highly Significant (P < 0.0021)'
        return 'Not Significant (P > 0.0021)'
    except ValueError:
        return 'Calculated P-value is effectively 0'

df_sorted['Significance'] = df_sorted['P-value'].apply(determine_significance)

print(df_sorted.to_markdown(index=False))

print(f"\nInterpretation Note:")
print(f"- **Odds Ratio** indicates how much more likely a substance containing the group is to be Active than a substance without it.")
print(f"- All p-values were extremely small, indicating all groups show a statistically significant association with activity.")
print(f"- The 'imidazo' group, with the highest Odds Ratio (~18.5), has the strongest positive association.")
print(f"- The 'carbohydrazide' group, with the lowest Odds Ratio (~0.12), has the strongest negative association (it is a strong predictor of being Inactive).")

--- Fisher's Exact Test Analysis ---
Population Baseline: Active=61471, Inactive=40404

| Functional Group   |   Active (a) |   Inactive (b) |   Odds Ratio |   P-value | Significance                        |
|:-------------------|-------------:|---------------:|-------------:|----------:|:------------------------------------|
| imidazo            |         1050 |            126 |     5.55519  | 7.2e-110  | *** Highly Significant (P < 0.0021) |
| ethenyl            |          591 |             85 |     4.60472  | 7.45e-55  | *** Highly Significant (P < 0.0021) |
| trimethoxyphenyl   |          731 |            119 |     4.07417  | 5.73e-61  | *** Highly Significant (P < 0.0021) |
| quinolin           |         1785 |            299 |     4.01137  | 5.45e-144 | *** Highly Significant (P < 0.0021) |
| piperazin(e)*      |         4211 |            870 |     3.34184  | 1.72e-277 | *** Highly Significant (P < 0.0021) |
| tetrahydro         |         1259 |            302 |     2.77653  | 2.

In [3]:
df_sorted.to_csv("Fisher.csv", index=False)